In [5]:
# Lab 4 (Step 4): Correlations in Data

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from pathlib import Path

# new data path ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
cwd = Path.cwd()
indoorpath = cwd / "datafolder" / "indoor.csv"
rundatapath = cwd / "datafolder" / "rundata.csv"
aq_inside_path = cwd / "datafolder" / "aq_temp_inside.csv"
aq_outside_path = cwd / "datafolder" / "aq_temp_outside.csv"
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

# -----------------------------
# Load data
# -----------------------------


indoor     = pd.read_csv(indoorpath)           # Our Group Indoor
rundata    = pd.read_csv(rundatapath)          # Our Group Outdoor
aq_inside  = pd.read_csv(aq_inside_path)   # Other Group Indoor
aq_inside = aq_inside.iloc[1:].reset_index(drop=True)  # drop bad first pressure datapoint
aq_outside = pd.read_csv(aq_outside_path)  # Other Group Outdoor

# -----------------------------
# Column mappings
# Compare ONLY these variables (disregard all others):
#   Other Group column   <->  Our Group column
#   Time                 <->  Timestamp
#   Temperature          <->  Temp
#   Humidity             <->  Humidity
#   Pressure             <->  Pressure
#   2.5                  <->  PM25_Env
# -----------------------------
PAIR_MAP = {
    "Temperature": "Temp",
    "Humidity": "Humidity",
    "Pressure": "Pressure",
    "2.5": "PM25_Env",
}

T_END = 300  # seconds

def prepare_df(df, time_col, keep_cols, t_end=T_END, gap_threshold=30):
    """Keep only required columns, build a relative time axis in seconds, and trim to [0, t_end].

    Notes:
    - Some logs include an early 'stray' record far away (in time) from the main run.
      To avoid chopping off the main run when we later trim to 0..t_end, we detect a
      large time gap and, if it occurs near the beginning, we start at the first sample
      *after* that gap.
    """
    df = df.copy()

    cols = [time_col] + [c for c in keep_cols if c in df.columns]
    df = df[cols]

    # Convert time to numeric (seconds-like) and drop non-numeric rows
    t = pd.to_numeric(df[time_col], errors="coerce")
    df = df.assign(t_raw=t).dropna(subset=["t_raw"]).copy()

    # Sort by time
    df = df.sort_values("t_raw").reset_index(drop=True)

    # Choose a robust t0
    t_raw = df["t_raw"].to_numpy()
    if len(t_raw) >= 2:
        dt = np.diff(t_raw)
        # Find the first large gap
        large_gap_idx = np.where(dt > gap_threshold)[0]
        if large_gap_idx.size > 0:
            j = int(large_gap_idx[0] + 1)  # first index after the gap
            # Only apply if the gap is near the beginning (likely a stray startup record)
            if j <= max(5, int(0.1 * len(df))):
                df = df.iloc[j:].reset_index(drop=True)
                t_raw = df["t_raw"].to_numpy()

    t0 = float(t_raw[0]) if len(t_raw) else np.nan
    df["t_sec"] = df["t_raw"] - t0

    # Coerce required data columns to numeric
    for c in keep_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # Trim to [0, t_end]
    df = df[(df["t_sec"] >= 0) & (df["t_sec"] <= t_end)].copy()
    return df

def drop_first_seconds_and_rezero(df, seconds, t_end=T_END):
    """Drop the first `seconds` of data (by t_sec) and re-zero time so plots run 0..t_end."""
    if seconds is None or seconds <= 0 or len(df) == 0:
        return df
    df = df[df["t_sec"] >= seconds].copy()
    df["t_sec"] = df["t_sec"] - seconds
    df = df[(df["t_sec"] >= 0) & (df["t_sec"] <= t_end)].copy()
    return df

def stats_table(x_a, x_b, label_a, label_b):
    x_a = np.asarray(x_a, dtype=float)
    x_b = np.asarray(x_b, dtype=float)
    x_a = x_a[np.isfinite(x_a)]
    x_b = x_b[np.isfinite(x_b)]

    def _stats(x):
        if len(x) == 0:
            return {"N": 0, "Mean": np.nan, "Std": np.nan, "Min": np.nan, "Max": np.nan}
        return {"N": len(x), "Mean": np.mean(x), "Std": np.std(x, ddof=1) if len(x) > 1 else 0.0,
                "Min": np.min(x), "Max": np.max(x)}

    a = _stats(x_a); b = _stats(x_b)
#     return pd.DataFrame([a, b], index=[label_a, label_b])  # removed per request


def stats_and_test(x_a, x_b):
    """Return a dict of descriptive stats + how many sigma apart the means are."""
    x_a = pd.to_numeric(pd.Series(x_a), errors="coerce").to_numpy(dtype=float)
    x_b = pd.to_numeric(pd.Series(x_b), errors="coerce").to_numpy(dtype=float)

    x_a = x_a[np.isfinite(x_a)]
    x_b = x_b[np.isfinite(x_b)]

    n_a, n_b = len(x_a), len(x_b)
    mean_a = np.mean(x_a) if n_a else np.nan
    mean_b = np.mean(x_b) if n_b else np.nan

    sigma_a = np.std(x_a, ddof=1) if n_a > 1 else np.nan
    sigma_b = np.std(x_b, ddof=1) if n_b > 1 else np.nan

    sigma_mean_a = sigma_a / np.sqrt(n_a) if n_a > 1 else np.nan
    sigma_mean_b = sigma_b / np.sqrt(n_b) if n_b > 1 else np.nan

    sigma_diff = np.sqrt(sigma_mean_a**2 + sigma_mean_b**2) if np.isfinite(sigma_mean_a) and np.isfinite(sigma_mean_b) else np.nan
    diff = abs(mean_a - mean_b) if np.isfinite(mean_a) and np.isfinite(mean_b) else np.nan
    n_sigma = (diff / sigma_diff) if np.isfinite(diff) and np.isfinite(sigma_diff) and sigma_diff != 0 else np.nan

    return {
        "n_a": n_a, "n_b": n_b,
        "mean_a": mean_a, "mean_b": mean_b,
        "sigma_a": sigma_a, "sigma_b": sigma_b,
        "sigma_mean_a": sigma_mean_a, "sigma_mean_b": sigma_mean_b,
        "diff": diff, "sigma_diff": sigma_diff,
        "n_sigma": n_sigma,
        "different_3sigma": bool(np.isfinite(n_sigma) and n_sigma > 3.0),
    }

def compare_one_variable(df_a, df_b, col_a, col_b, label_a, label_b, env_label, var_label):
    """Plot time series + histogram for one variable (Our Group vs Other Group)."""
    # Time series
    plt.figure()
    plt.plot(df_a["t_sec"], df_a[col_a], label=label_a)
    plt.plot(df_b["t_sec"], df_b[col_b], label=label_b)
    plt.xlim(0, T_END)
    plt.xlabel("Time (s)")
    plt.ylabel(var_label)
    plt.title(f"Our Group vs Other Group {env_label} {var_label} (Time Series)")
    plt.legend()
    plt.grid(True, alpha=0.2)
    plt.show()

    # Histogram
    plt.figure()
    plt.hist(df_a[col_a].dropna(), bins=30, alpha=0.6, label=label_a)
    plt.hist(df_b[col_b].dropna(), bins=30, alpha=0.6, label=label_b)
    plt.xlabel(var_label)
    plt.ylabel("Count")
    plt.title(f"Our Group vs Other Group {env_label} {var_label} (Histogram)")
    plt.legend()
    plt.grid(True, alpha=0.2)
    plt.show()
    # Summary stats table
#     display(stats_table(df_a[col_a], df_b[col_b], label_a, label_b))  # removed per request

    # Full stats + sigma separation (matches the other notebook)
    s = stats_and_test(df_a[col_a], df_b[col_b])
    print(f"--- {env_label} {var_label} ---")
    print(f"N {label_a} = {s['n_a']}, N {label_b} = {s['n_b']}")
    print(f"Mean {label_a}  = {s['mean_a']:.6g}")
    print(f"Mean {label_b} = {s['mean_b']:.6g}")
    print(f"Sigma (intrinsic) {label_a}  = {s['sigma_a']:.6g}")
    print(f"Sigma (intrinsic) {label_b} = {s['sigma_b']:.6g}")
    print(f"Sigma_mean {label_a}  = {s['sigma_mean_a']:.6g}")
    print(f"Sigma_mean {label_b} = {s['sigma_mean_b']:.6g}")
    print(f"|Δmean| = {s['diff']:.6g}")
    print(f"Combined sigma_mean (for Δ) = {s['sigma_diff']:.6g}")
    print(f"Separation (|Δmean| / σΔmean) = {s['n_sigma']:.6g} σ")
    if s["different_3sigma"]:
        print("Statistically different at > 3σ")
    else:
        print("Not statistically different at > 3σ")
# -----------------------------
# Prepare datasets
# -----------------------------
our_indoor   = prepare_df(indoor,     time_col="Timestamp", keep_cols=list(PAIR_MAP.values()))
other_indoor = prepare_df(aq_inside,  time_col="Time",      keep_cols=list(PAIR_MAP.keys()))

our_outdoor   = prepare_df(rundata,    time_col="Timestamp", keep_cols=list(PAIR_MAP.values()))
other_outdoor = prepare_df(aq_outside, time_col="Time",      keep_cols=list(PAIR_MAP.keys()))

# Mask off first 30 seconds for OUTDOOR comparison (from each dataset), then re-zero time to start at 0s
our_outdoor   = drop_first_seconds_and_rezero(our_outdoor,   30)
other_outdoor = drop_first_seconds_and_rezero(other_outdoor, 30)

LABEL_OUR_INDOOR   = "Our Group Indoor"
LABEL_OTHER_INDOOR = "Other Group Indoor"
LABEL_OUR_OUTDOOR  = "Our Group Outdoor"
LABEL_OTHER_OUTDOOR= "Other Group Outdoor"


FileNotFoundError: [Errno 2] No such file or directory: 'indoor.csv'

In [ ]:
#our groups indoor humidity + temp. and another group's humidity + temp data plotted 

plt.figure()

plt.scatter(our_indoor["Temp"], our_indoor["Humidity"], label="Our Indoor")
plt.scatter(other_indoor["Temperature"], other_indoor["Humidity"], label="Other Indoor")

plt.xlabel("Temperature")
plt.ylabel("Humidity")
plt.title("Temperature vs Humidity")

plt.legend()
plt.grid(True)

plt.show()

In [7]:
#Hidden variable? --> TIME! Time needs to be equal at each measurement to properly do comparison.#

In [8]:
#This matters because if we try to graph two sets of data and their times aren't equal at each point we're comparing, the difference could be caused by time and not actually the variables we're comparing.